# Adaptive RAG — Corrected & Rebuilt

This notebook fixes the structural bugs in the original version and implements
**genuine** Adaptive RAG (query-complexity routing), not just a hybrid
web+memory retriever.

**What changed vs. the original notebook:**
1. **Real adaptivity** — each question is routed to one of three strategies based on
   estimated complexity: *no retrieval*, *single-hop retrieval*, or *multi-hop
   iterative retrieval* — matching the actual Adaptive-RAG idea (Jeong et al., 2024).
2. **A dataset that actually needs adaptivity** — a mix of simple factoid questions
   (TriviaQA, no-context), single-hop questions with a supporting passage (SQuAD),
   and multi-hop questions (HotpotQA). A dataset made of only multi-hop questions
   (like the original notebook's HotpotQA-only setup) can never show an adaptive
   router's advantage, because every question would route the same way.
3. **A fair baseline** — same LLM, same prompt skeleton, same knowledge corpus.
   The only difference from Adaptive RAG is that the baseline always does exactly
   one retrieval step. This isolates the effect of *adaptivity* instead of
   confounding it with extra data sources (the original mixed in live web search
   only for the "adaptive" arm).
4. **No circular retrieval** — the corpus is built once from real source passages,
   never from the model's own previous answers (the original notebook's "memory"
   store did this, which lets the model reinforce its own earlier mistakes).
5. **One config, one checkpoint path, one model definition** — no more re-defining
   `DRIVE_PATH` five different ways or reinstalling packages eight times.
6. **Efficiency is measured, not just accuracy** — the real payoff of Adaptive RAG
   is answering easy questions with zero or one retrieval call instead of
   multi-hop search every time. We log retrieval-call counts per question and
   report them alongside ROUGE-L / BERTScore.


## 1. Setup

In [ ]:
!pip install -q -U \
  faiss-cpu \
  sentence-transformers \
  transformers \
  accelerate \
  bitsandbytes \
  langchain-core \
  langchain-text-splitters \
  langchain-huggingface \
  langchain-community \
  datasets \
  rouge-score \
  bert-score \
  tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ============================================================================
# SINGLE SOURCE OF TRUTH FOR CONFIG — do not redefine these later in the notebook.
# ============================================================================
import os, torch

DRIVE_PATH = "/content/drive/MyDrive/adaptive_rag_project"
os.makedirs(DRIVE_PATH, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CHECKPOINT   = "mistralai/Mistral-7B-Instruct-v0.2"   # one model, used everywhere
EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE, CHUNK_OVERLAP = 500, 50

N_PER_TIER = 120          # questions per complexity tier -> ~360 total eval questions
TOP_K = 3                 # passages retrieved per hop
MAX_HOPS = 2              # cap on multi-hop iterations

RESULTS_PATH = os.path.join(DRIVE_PATH, "results.csv")
SAVE_EVERY = 20

print(f"Device: {DEVICE}")
print(f"All artifacts will live under: {DRIVE_PATH}")


## 2. Build a mixed-complexity evaluation set

Adaptive RAG only has something to demonstrate if questions differ in how much
retrieval they need. We build three tiers:

| Tier | Source | Why it belongs here |
|---|---|---|
| **A — no retrieval needed** | TriviaQA (`rc.nocontext`) | Well-known facts a strong LLM usually already knows; retrieval adds latency without adding accuracy. |
| **B — single-hop retrieval** | SQuAD v1.1 | Answer is stated directly in one passage — one retrieval call is enough. |
| **C — multi-hop retrieval** | HotpotQA (distractor) | Answer requires combining facts across 2+ passages — needs iterative retrieval. |

Each tier also contributes its passages to a **single shared FAISS corpus**, so
retrieval for the whole notebook is happening against one consistent knowledge base.


In [ ]:
import re
import pandas as pd
from datasets import load_dataset
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

random_seed = 42

# --- Tier A: simple factoid, no context needed ---------------------------
trivia = load_dataset("trivia_qa", "rc.nocontext", split="validation")
trivia = trivia.shuffle(seed=random_seed).select(range(N_PER_TIER))

tier_a_rows = [{
    "question": ex["question"],
    "gold_answer": ex["answer"]["value"],
    "complexity_gold": "A_no_retrieval",
    "source_passages": [],   # nothing to add to the corpus
} for ex in trivia]

# --- Tier B: single-hop, needs exactly one supporting passage -------------
squad = load_dataset("squad", split="validation")
squad = squad.shuffle(seed=random_seed).select(range(N_PER_TIER))

tier_b_rows, squad_docs = [], []
for ex in squad:
    tier_b_rows.append({
        "question": ex["question"],
        "gold_answer": ex["answers"]["text"][0] if ex["answers"]["text"] else "",
        "complexity_gold": "B_single_hop",
        "source_passages": [ex["context"]],
    })
    squad_docs.append(Document(page_content=ex["context"], metadata={"src": "squad"}))

# --- Tier C: multi-hop, needs 2+ supporting passages -----------------------
hotpot = load_dataset("hotpot_qa", "distractor", split="validation", trust_remote_code=True)
hotpot = hotpot.shuffle(seed=random_seed).select(range(N_PER_TIER))

tier_c_rows, hotpot_docs = [], []
for ex in hotpot:
    passages = ["".join(sents) for sents in ex["context"]["sentences"]]
    tier_c_rows.append({
        "question": ex["question"],
        "gold_answer": ex["answer"],
        "complexity_gold": "C_multi_hop",
        "source_passages": passages,
    })
    for p in passages:
        hotpot_docs.append(Document(page_content=p, metadata={"src": "hotpotqa"}))

eval_df = pd.DataFrame(tier_a_rows + tier_b_rows + tier_c_rows).sample(
    frac=1.0, random_state=random_seed
).reset_index(drop=True)

print(eval_df["complexity_gold"].value_counts())
print(f"Total evaluation questions: {len(eval_df)}")
eval_df.head()


In [ ]:
# ---------------------------------------------------------------------------
# Build ONE shared knowledge corpus (chunked) — this is what BOTH the
# baseline and the adaptive pipeline retrieve from. No corpus is ever built
# from the model's own generated answers.
# ---------------------------------------------------------------------------
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
all_source_docs = squad_docs + hotpot_docs
corpus_chunks = splitter.split_documents(all_source_docs)

print(f"Source passages: {len(all_source_docs)} -> chunks: {len(corpus_chunks)}")


## 3. Embeddings + FAISS index (built once)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL, model_kwargs={"device": DEVICE})

FAISS_INDEX_DIR = os.path.join(DRIVE_PATH, "faiss_index")
if os.path.exists(FAISS_INDEX_DIR):
    print("Loading existing FAISS index...")
    vectorstore = FAISS.load_local(FAISS_INDEX_DIR, embeddings, allow_dangerous_deserialization=True)
else:
    print("Building FAISS index from the shared corpus...")
    vectorstore = FAISS.from_documents(corpus_chunks, embeddings)
    vectorstore.save_local(FAISS_INDEX_DIR)

retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
print(f"Corpus vectors: {vectorstore.index.ntotal}")


## 4. Load the LLM (one model, one place, quantized for a T4/A100)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

gen_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=64,       # answers are short factoid strings, not essays
    do_sample=False,         # deterministic decoding -> fair, reproducible comparison
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
)
llm = HuggingFacePipeline(pipeline=gen_pipeline)
print("LLM ready.")


## 5. Query-complexity router

The original Adaptive-RAG paper trains a small classifier on labeled data. As a
practical, dependency-light stand-in, we use the LLM itself with a constrained
few-shot prompt to output exactly one label. This is a simplification — swap in
a trained classifier (e.g. a fine-tuned DistilBERT on complexity labels) for
production use — but it is enough to demonstrate real routing behavior, and it
costs one short generation instead of a full answer generation when it predicts 'A'.


In [ ]:
ROUTER_PROMPT = '''Classify the QUESTION below into exactly one category. Answer with only the single letter.

A = a simple, well-known fact a knowledgeable person could answer from memory, no lookup needed.
B = the answer likely requires looking up ONE specific document or fact.
C = the answer requires combining information from TWO OR MORE different sources (comparisons, multi-step reasoning, "who was X when Y happened", etc).

Examples:
Q: What is the capital of France?
A

Q: What year did the Eiffel Tower open to the public?
B

Q: Which university did the director of the 1997 film that won Best Picture attend?
C

Q: {question}
'''.strip()

def classify_complexity(question: str) -> str:
    prompt = ROUTER_PROMPT.format(question=question)
    raw = llm.invoke(prompt).strip()
    match = re.search(r"\b([ABC])\b", raw.upper())
    return match.group(1) if match else "B"   # default to single-hop if unclear


## 6. Answer strategies

In [ ]:
ANSWER_PROMPT = '''Answer the question in a single short phrase (a few words), with no extra explanation.
{context_block}
Question: {question}
Answer:'''

def format_context(docs):
    return "\n".join(d.page_content for d in docs)

def strategy_no_retrieval(question):
    prompt = ANSWER_PROMPT.format(context_block="", question=question)
    return llm.invoke(prompt).strip(), 0

def strategy_single_hop(question):
    docs = retriever.invoke(question)
    context_block = f"Context:\n{format_context(docs)}\n"
    prompt = ANSWER_PROMPT.format(context_block=context_block, question=question)
    return llm.invoke(prompt).strip(), 1

def strategy_multi_hop(question, max_hops=MAX_HOPS):
    '''Iterative retrieval: retrieve, ask the model whether it has enough info
    and what to search for next, retrieve again if needed, then answer.'''
    gathered_docs, retrieval_calls = [], 0
    current_query = question

    for hop in range(max_hops):
        docs = retriever.invoke(current_query)
        gathered_docs.extend(docs)
        retrieval_calls += 1

        followup_prompt = f'''You are answering a multi-hop question step by step.
Original question: {question}
Evidence gathered so far:
{format_context(gathered_docs)}

If the evidence above is enough to answer, respond with exactly: DONE
Otherwise, respond with ONE short follow-up search query needed to find the missing fact.'''
        followup = llm.invoke(followup_prompt).strip()

        if followup.upper().startswith("DONE") or hop == max_hops - 1:
            break
        current_query = followup

    context_block = f"Context:\n{format_context(gathered_docs)}\n"
    prompt = ANSWER_PROMPT.format(context_block=context_block, question=question)
    return llm.invoke(prompt).strip(), retrieval_calls

STRATEGY_MAP = {"A": strategy_no_retrieval, "B": strategy_single_hop, "C": strategy_multi_hop}

def adaptive_rag_answer(question):
    label = classify_complexity(question)
    answer, n_retrievals = STRATEGY_MAP[label](question)
    return answer, label, n_retrievals

def baseline_rag_answer(question):
    '''Fair baseline: always exactly one retrieval step, same prompt skeleton,
    same LLM. The only difference from adaptive RAG is the absence of routing.'''
    answer, n_retrievals = strategy_single_hop(question)
    return answer, n_retrievals


## 7. Run both pipelines with checkpointing (single, correct resume logic)

In [ ]:
def run_pipeline(df, answer_fn, is_adaptive, out_path):
    if os.path.exists(out_path):
        done_df = pd.read_csv(out_path)
        start = len(done_df)
        rows = done_df.to_dict("records")
        print(f"Resuming '{out_path}' from row {start}/{len(df)}")
    else:
        rows, start = [], 0
        print(f"Starting '{out_path}' from scratch")

    for i in tqdm(range(start, len(df)), initial=start, total=len(df)):
        ex = df.iloc[i]
        try:
            if is_adaptive:
                answer, predicted_label, n_retrievals = answer_fn(ex["question"])
            else:
                answer, n_retrievals = answer_fn(ex["question"])
                predicted_label = None
        except Exception as e:
            answer, predicted_label, n_retrievals = f"ERROR: {e}", None, -1

        rows.append({
            "question": ex["question"],
            "gold_answer": ex["gold_answer"],
            "complexity_gold": ex["complexity_gold"],
            "predicted_label": predicted_label,
            "answer": answer,
            "n_retrievals": n_retrievals,
        })

        if (i + 1) % SAVE_EVERY == 0 or (i + 1) == len(df):
            pd.DataFrame(rows).to_csv(out_path, index=False)

    return pd.DataFrame(rows)


from tqdm.auto import tqdm

BASELINE_PATH = os.path.join(DRIVE_PATH, "baseline_results.csv")
ADAPTIVE_PATH = os.path.join(DRIVE_PATH, "adaptive_results.csv")

baseline_df = run_pipeline(eval_df, baseline_rag_answer, is_adaptive=False, out_path=BASELINE_PATH)
adaptive_df = run_pipeline(eval_df, adaptive_rag_answer, is_adaptive=True, out_path=ADAPTIVE_PATH)


## 8. Evaluation — accuracy AND efficiency

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore_score
import numpy as np

def add_scores(df, answer_col, prefix):
    df[answer_col] = df[answer_col].fillna("").astype(str)
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    df[f"{prefix}_rouge_l"] = [
        scorer.score(g, a)["rougeL"].fmeasure
        for g, a in zip(df["gold_answer"].astype(str), df[answer_col])
    ]
    _, _, f1 = bertscore_score(df[answer_col].tolist(), df["gold_answer"].astype(str).tolist(),
                                lang="en", verbose=False)
    df[f"{prefix}_bertscore"] = f1.numpy()
    return df

baseline_df = add_scores(baseline_df, "answer", "baseline")
adaptive_df = add_scores(adaptive_df, "answer", "adaptive")

merged = baseline_df[["question", "gold_answer", "complexity_gold", "answer", "n_retrievals",
                       "baseline_rouge_l", "baseline_bertscore"]].rename(
    columns={"answer": "baseline_answer", "n_retrievals": "baseline_retrievals"}
).merge(
    adaptive_df[["question", "predicted_label", "answer", "n_retrievals",
                 "adaptive_rouge_l", "adaptive_bertscore"]].rename(
        columns={"answer": "adaptive_answer", "n_retrievals": "adaptive_retrievals"}
    ),
    on="question", how="inner",
)
merged.to_csv(RESULTS_PATH, index=False)
merged.head()


In [ ]:
summary = pd.DataFrame({
    "Metric": ["ROUGE-L (F1)", "BERTScore (F1)", "Avg. retrieval calls / question"],
    "Baseline RAG": [
        merged["baseline_rouge_l"].mean(),
        merged["baseline_bertscore"].mean(),
        merged["baseline_retrievals"].mean(),
    ],
    "Adaptive RAG": [
        merged["adaptive_rouge_l"].mean(),
        merged["adaptive_bertscore"].mean(),
        merged["adaptive_retrievals"].mean(),
    ],
}).round(4)

print("=== Overall ===")
display(summary)

print("\n=== Accuracy by TRUE complexity tier (does routing help where it should?) ===")
per_tier = merged.groupby("complexity_gold")[
    ["baseline_rouge_l", "adaptive_rouge_l", "baseline_bertscore", "adaptive_bertscore",
     "baseline_retrievals", "adaptive_retrievals"]
].mean().round(4)
display(per_tier)

print("\n=== Router accuracy (predicted vs. gold complexity label) ===")
router_acc = (merged["predicted_label"].str[0] == merged["complexity_gold"].str[0]).mean()
print(f"Router agreement with gold tier: {router_acc:.2%}")


## 9. Plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = ["ROUGE-L (F1)", "BERTScore (F1)"]
x = np.arange(len(metrics))
width = 0.35
axes[0].bar(x - width/2, summary.loc[[0,1], "Baseline RAG"], width, label="Baseline RAG", color="skyblue")
axes[0].bar(x + width/2, summary.loc[[0,1], "Adaptive RAG"], width, label="Adaptive RAG", color="lightcoral")
axes[0].set_xticks(x); axes[0].set_xticklabels(metrics)
axes[0].set_ylim(0, 1.0); axes[0].set_ylabel("F1 Score")
axes[0].set_title("Accuracy: Baseline vs. Adaptive RAG")
axes[0].legend()

axes[1].bar(["Baseline RAG", "Adaptive RAG"],
            [merged["baseline_retrievals"].mean(), merged["adaptive_retrievals"].mean()],
            color=["skyblue", "lightcoral"])
axes[1].set_ylabel("Avg. retrieval calls / question")
axes[1].set_title("Efficiency: Retrieval calls per question")

fig.tight_layout()
plt.show()


In [ ]:
tiers = per_tier.index.tolist()
x = np.arange(len(tiers))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, per_tier["baseline_bertscore"], width, label="Baseline RAG", color="skyblue")
ax.bar(x + width/2, per_tier["adaptive_bertscore"], width, label="Adaptive RAG", color="lightcoral")
ax.set_xticks(x); ax.set_xticklabels(tiers)
ax.set_ylabel("BERTScore (F1)")
ax.set_title("BERTScore by question-complexity tier")
ax.legend()
fig.tight_layout()
plt.show()


## Notes on interpreting results

- Look at the **per-tier table**, not just the overall average. A well-behaved
  adaptive router should roughly match baseline accuracy on tier A/B while using
  **fewer retrieval calls** on tier A, and should **beat** the baseline on tier C
  where multi-hop retrieval actually helps.
- If the router's agreement with the gold tier is low, the LLM-prompt classifier
  is the bottleneck — replace it with a small trained classifier (e.g. fine-tune
  DistilBERT on complexity labels derived from whether a single-hop retriever
  alone gets each training question right, per the original Adaptive-RAG paper's
  labeling procedure) before trusting the comparison.
- Because both pipelines share one corpus, one LLM, and one prompt skeleton, any
  remaining gap is attributable to the *routing decision itself* — that's the
  fair comparison the original notebook was missing.
